# 22 · Preparacion de datos — QUESTIONNAIRE

**Orden (igual que prueba2, parando en encoding):**
```
Limpieza -> Imputacion (escalado + KNN) -> Encoding
```
Usa los mismos nodos Kedro (`src/nhanes/pipelines/...`) con el bloque `quest` de `parameters.yml`.

- **Input:** `data/02_intermediate/quest_intermediate.parquet` (SEQN = indice)
- **Target reservado:** salud_autopercibida (1=excelente ... 5=mala), fuerte predictor de mortalidad
- **Nota:** `tabaquismo` y `sueno_categoria` -> 'Unknown' + One-Hot. `missing_threshold=0.6` elimina dialisis_flag, cigarrillos_dia, medicacion_presion, limitacion_funcional, tragos_por_dia (todas condicionales/leakage).
- **Salida:** `data/05_model_input/quest_encoded.parquet`

> El entrenamiento va despues.

In [1]:
import sys, yaml
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from nhanes.pipelines.data_cleaning.nodes import clean_dataset
from nhanes.pipelines.data_imputation.nodes import impute_dataset
from nhanes.pipelines.data_encoding.nodes import encode_dataset

params = yaml.safe_load(open(ROOT / 'conf/base/parameters.yml', encoding='utf-8'))['quest']
print('Parametros quest:'); print(yaml.dump(params, allow_unicode=True, sort_keys=False))

df = pd.read_parquet(ROOT / 'data/02_intermediate/quest_intermediate.parquet')
print('Intermediate:', df.shape)
df.head()

Parametros quest:
cols_to_drop: []
target_cols:
- salud_autopercibida
missing_threshold: 0.6
categorical_cols_impute:
- tabaquismo
- sueno_categoria
scaler_type: standard
knn_neighbors: 5
encoding:
  binary_cols: []
  ordinal_maps: {}
  nominal_cols:
  - tabaquismo
  - sueno_categoria

Intermediate: (9254, 29)


,tabaquismo,cigarrillos_dia,alcohol_freq_anual,tragos_por_dia,salud_autopercibida,met_min_semana,activo_oms,sedentario_min,limitacion_funcional,phq9_score,...,hipertension_dx,medicacion_presion,colesterol_alto_dx,medicacion_colesterol,enf_renal_dx,dialisis_flag,n_medicamentos,polifarmacia_flag,cambio_peso_1y_lb,discapacidad_count
SEQN,,,,,,,,,,,,,,,,,,,,,
93705,exfumador,NaN,7.0,1.0,3.0,480.0,0,300.0,0.0,0.0,...,1.0,1.0,0.0,0.0,0.0,NaN,3.0,0,0.0,0.0
93706,nunca,NaN,NaN,NaN,2.0,1140.0,1,240.0,NaN,0.0,...,0.0,NaN,0.0,0.0,NaN,NaN,0.0,0,0.0,0.0
93707,NaN,NaN,NaN,NaN,3.0,NaN,<NA>,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0,NaN,0.0
93708,nunca,NaN,NaN,NaN,3.0,600.0,1,120.0,7.0,0.0,...,1.0,1.0,1.0,1.0,0.0,NaN,3.0,0,0.0,0.0
93709,activo,5.0,NaN,NaN,NaN,1440.0,1,600.0,16.0,NaN,...,1.0,1.0,0.0,0.0,0.0,NaN,4.0,0,-10.0,2.0


## 1. Limpieza (`data_cleaning`)

Elimina `cols_to_drop`, **reserva el target**, poda features con missingness > `missing_threshold`, y rellena categoricas faltantes con `'Unknown'`.

In [2]:
print('Nulos por columna ANTES:')
print(df.isnull().mean().round(3).sort_values(ascending=False).to_string())

quest_clean, quest_target_raw = clean_dataset(df, params)
print()
print('Features limpias:', quest_clean.shape)
print(list(quest_clean.columns))
print('Target reservado:', list(quest_target_raw.columns))
quest_target_raw.describe().round(2)

Nulos por columna ANTES:
dialisis_flag            0.976
cigarrillos_dia          0.890
medicacion_presion       0.769
limitacion_funcional     0.685
tragos_por_dia           0.623
medicacion_colesterol    0.510
alcohol_freq_anual       0.509
depresion_flag           0.450
phq9_score               0.450
artritis_flag            0.400
enf_renal_dx             0.399
cancer_flag              0.398
cvd_flag                 0.398
comorbilidad_count       0.398
met_min_semana           0.372
sedentario_min           0.372
tabaquismo               0.367
activo_oms               0.367
salud_autopercibida      0.356
cambio_peso_1y_lb        0.352
horas_sueno              0.339
sueno_categoria          0.339
colesterol_alto_dx       0.339
hipertension_dx          0.335
discapacidad_count       0.129
diabetes_flag            0.039
diabetes_insulina        0.039
polifarmacia_flag        0.001
n_medicamentos           0.001

Features limpias: (9254, 23)
['tabaquismo', 'alcohol_freq_anual', 'met_min_

,salud_autopercibida
count,5964.00
mean,2.79
std,0.97
min,1.00
25%,2.00
50%,3.00
75%,3.00
max,5.00


## 2. Imputacion (`data_imputation`)

`StandardScaler` y luego **KNN (k=5)** sobre los datos ya escalados (distancias comparables).

In [3]:
print('Nulos ANTES:', int(quest_clean.isnull().sum().sum()))
quest_imputed = impute_dataset(quest_clean, params)
print('Nulos DESPUES:', int(quest_imputed.isnull().sum().sum()))
quest_imputed.describe().round(3)

Nulos ANTES: 61079


Nulos DESPUES: 0


,alcohol_freq_anual,met_min_semana,activo_oms,sedentario_min,phq9_score,depresion_flag,horas_sueno,comorbilidad_count,cvd_flag,cancer_flag,...,diabetes_flag,diabetes_insulina,hipertension_dx,colesterol_alto_dx,medicacion_colesterol,enf_renal_dx,n_medicamentos,polifarmacia_flag,cambio_peso_1y_lb,discapacidad_count
count,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,...,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000,9254.000
mean,-0.014,0.136,0.188,-0.054,-0.114,-0.089,0.060,-0.171,-0.140,-0.106,...,-0.008,-0.007,-0.071,-0.157,-0.243,-0.074,-0.000,-0.000,0.046,-0.051
std,0.738,0.869,0.874,0.848,0.795,0.784,0.830,0.818,0.800,0.799,...,0.983,0.981,0.860,0.876,0.782,0.786,1.000,1.000,0.875,0.945
min,-1.389,-0.600,-1.250,-1.666,-0.764,-0.316,-3.389,-0.663,-0.376,-0.344,...,-0.334,-0.171,-0.730,-0.688,-0.758,-0.204,-0.584,-0.379,-7.764,-0.483
25%,-0.258,-0.515,-0.840,-0.521,-0.575,-0.316,-0.395,-0.663,-0.376,-0.344,...,-0.334,-0.171,-0.730,-0.688,-0.758,-0.204,-0.584,-0.379,-0.364,-0.483
50%,0.025,-0.124,0.800,-0.159,-0.293,-0.316,0.085,-0.496,-0.376,-0.344,...,-0.334,-0.171,-0.730,-0.688,-0.758,-0.204,-0.584,-0.379,-0.028,-0.483
75%,0.308,0.782,0.800,0.504,-0.058,-0.316,0.504,0.169,-0.376,-0.344,...,-0.334,-0.171,0.110,0.168,-0.343,-0.204,0.164,-0.379,0.566,-0.288
max,1.439,7.634,0.800,4.963,5.120,3.167,3.798,5.995,2.657,2.910,...,2.993,5.833,1.371,1.453,1.319,4.892,7.645,2.637,13.425,5.381


## 3. Encoding (`data_encoding`)

Label / Ordinal / One-Hot segun el bloque `encoding`. Resultado totalmente numerico.

In [4]:
quest_encoded = encode_dataset(quest_imputed, params)
print('Shape final:', quest_encoded.shape)
print('dtypes:', set(quest_encoded.dtypes.astype(str)))
print('Nulos:', int(quest_encoded.isnull().sum().sum()), '| indice:', quest_encoded.index.name)
print('Columnas:', list(quest_encoded.columns))
quest_encoded.head()

Shape final: (9254, 29)
dtypes: {'int64', 'float64'}
Nulos: 0 | indice: SEQN
Columnas: ['alcohol_freq_anual', 'met_min_semana', 'activo_oms', 'sedentario_min', 'phq9_score', 'depresion_flag', 'horas_sueno', 'comorbilidad_count', 'cvd_flag', 'cancer_flag', 'artritis_flag', 'diabetes_flag', 'diabetes_insulina', 'hipertension_dx', 'colesterol_alto_dx', 'medicacion_colesterol', 'enf_renal_dx', 'n_medicamentos', 'polifarmacia_flag', 'cambio_peso_1y_lb', 'discapacidad_count', 'tabaquismo_Unknown', 'tabaquismo_activo', 'tabaquismo_exfumador', 'tabaquismo_nunca', 'sueno_categoria_Unknown', 'sueno_categoria_corto', 'sueno_categoria_largo', 'sueno_categoria_normal']


,alcohol_freq_anual,met_min_semana,activo_oms,sedentario_min,phq9_score,depresion_flag,horas_sueno,comorbilidad_count,cvd_flag,cancer_flag,...,cambio_peso_1y_lb,discapacidad_count,tabaquismo_Unknown,tabaquismo_activo,tabaquismo_exfumador,tabaquismo_nunca,sueno_categoria_Unknown,sueno_categoria_corto,sueno_categoria_largo,sueno_categoria_normal
SEQN,,,,,,,,,,,,,,,,,,,,,
93705,0.590783,-0.531817,-1.250120,-0.159090,-0.763617,-0.315715,0.204339,0.169460,-0.376411,-0.343651,...,-0.028117,-0.483215,0,0,1,0,0,0,0,1
93706,0.590783,-0.438634,0.799923,-0.460399,-0.763617,-0.315715,1.701731,-0.662747,-0.376411,-0.343651,...,-0.028117,-0.483215,0,0,0,1,0,0,1,0
93707,0.025101,0.781777,0.799923,-0.520661,-0.245846,-0.315715,0.084548,-0.329864,-0.376411,-0.343651,...,0.566084,-0.483215,1,0,0,0,1,0,0,0
93708,-0.823421,-0.514875,0.799923,-1.063016,-0.763617,-0.315715,0.204339,0.169460,-0.376411,-0.343651,...,-0.028117,-0.483215,0,0,0,1,0,0,0,1
93709,-0.144603,-0.396278,0.799923,1.347452,0.554347,0.380912,-0.394618,1.001667,2.656668,-0.343651,...,-0.588684,1.471482,0,1,0,0,0,0,0,1


## 4. Guardado

In [5]:
for d in ['03_primary','04_feature','05_model_input']:
    (ROOT / 'data' / d).mkdir(parents=True, exist_ok=True)
quest_clean.to_parquet(ROOT / 'data/03_primary/quest_clean.parquet')
quest_target_raw.to_parquet(ROOT / 'data/03_primary/quest_target_raw.parquet')
quest_imputed.to_parquet(ROOT / 'data/04_feature/quest_imputed.parquet')
quest_encoded.to_parquet(ROOT / 'data/05_model_input/quest_encoded.parquet')
print('Guardado OK')

Guardado OK

## Conclusiones — QUESTIONNAIRE

- Target = salud_autopercibida (reservada). Las columnas con >60% de nulos (condicionales) se eliminan; el resto se imputa con KNN. Categoricas codificadas con One-Hot incluyendo 'Unknown'.
- Salida `quest_encoded`: escalado, sin nulos, `SEQN` como indice para unir con DEMO al final.
- **Pendiente (modelado):** construir target de clasificacion, split y entrenar (reg / clf / no supervisado).